# ChuckleNet: Scale to 1000+ Videos
## Pipeline: Download → Extract → Label → Train

**Runtime**: ~4-6 hours on Colab T4 GPU
**Output**: Fusion model trained on 500-1000+ videos

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/chuckle_net')

# Install dependencies
!pip install -q librosa numpy pandas scikit-learn torch transformers faster-whisper

import numpy as np
import pandas as pd
import glob, os, json
from pathlib import Path

BASE = '/content/drive/MyDrive/chuckle_net'
DATA_DIR = f'{BASE}/data_1000'
os.makedirs(DATA_DIR, exist_ok=True)
print(f'Data dir: {DATA_DIR}')

In [ ]:
# === STEP 1: Download Comedy Videos ===
# Using yt-dlp with browser cookies for authentication

!pip install -q yt-dlp

# Search for comedy videos
import subprocess, json

def search_youtube(query, n=50):
    """Search YouTube for videos"""
    cmd = ['python3', '-m', 'yt_dlp', f'--playlist-end', str(n),
            '--flat-playlist', '--print', '%(id)s|%(duration)s|%(title)s',
            f'ytsearch:{query}']
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    videos = []
    for line in r.stdout.strip().split('\n'):
        if '|' not in line: continue
        parts = line.split('|')
        if len(parts) >= 3:
            vid, dur, title = parts[0], parts[1], parts[2]
            try:
                dur = int(dur)
                if dur > 600:  # >10 min
                    videos.append((vid, title, dur))
            except: pass
    return videos

print('Searching for comedy videos...')
all_vids = {}
for q in ['stand up comedy full special 2024', 'comedy special full episode', 
           'hindi comedy special full', 'tamil standup comedy', 
           'chinese comedy full show', 'korean stand up']:
    vids = search_youtube(q, 30)
    for v, t, d in vids:
        if v not in all_vids:
            all_vids[v] = (t, d)
    print(f'  {q}: {len(vids)} videos')

print(f'\nTotal unique videos: {len(all_vids)}')

# Save video list
with open(f'{DATA_DIR}/video_list.json', 'w') as f:
    json.dump({v: {'title': t, 'duration': d} for v, (t, d) in all_vids.items()}, f)

# Download (limit to 500 to stay within disk space)
import subprocess
vids_to_download = list(all_vids.keys())[:500]
print(f'Downloading {len(vids_to_download)} videos...')

downloaded, failed = [], []
for i, vid in enumerate(vids_to_download):
    out_file = f'{DATA_DIR}/audio/{vid}.m4a'
    os.makedirs(f'{DATA_DIR}/audio', exist_ok=True)
    if os.path.exists(out_file):
        downloaded.append(vid)
        continue
    try:
        r = subprocess.run(
            ['python3', '-m', 'yt_dlp', '-f', 'bestaudio[ext=m4a]',
             '-o', out_file, '--no-warnings',
             f'https://youtube.com/watch?v={vid}'],
            capture_output=True, timeout=60
        )
        if os.path.exists(out_file):
            downloaded.append(vid)
        else:
            failed.append(vid)
    except:
        failed.append(vid)
    
    if (i+1) % 50 == 0:
        print(f'  [{i+1}/{len(vids_to_download)}] OK={len(downloaded)} FAIL={len(failed)}')

print(f'\nDownloaded: {len(downloaded)}, Failed: {len(failed)}')
print(f'Audio files: {len(glob.glob(f"{DATA_DIR}/audio/*.m4a"))}')

In [ ]:
# === STEP 2: Extract Prosody Features (CPU - FAST) ===
# 23-dim prosody: F0, energy, ZCR, spectral, MFCCs

import librosa, numpy as np, os
from tqdm import tqdm

def extract_prosody(audio_path, sr=22050):
    """Extract 23-dim prosody features"""
    try:
        y, sr = librosa.load(audio_path, sr=sr)
        if len(y) < sr: return None
        
        # Energy
        rms = librosa.feature.rms(y=y, hop_length=512)[0]
        
        # Pitch (F0) - only voiced frames
        f0, voiced_flag, voiced_prob = librosa.pyin(
            y, fmin=50, fmax=500, sr=sr, hop_length=512
        )
        f0 = np.nan_to_num(f0, nan=0)
        voiced_f0 = f0[voiced_flag] if voiced_flag.any() else np.array([0])
        
        # Other features
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=512)[0]
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=512)[0]
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=512)[0]
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=512)[0]
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=512)
        
        return [
            np.mean(rms), np.std(rms), np.max(rms),  # Energy
            np.mean(zcr), np.std(zcr),  # ZCR
            np.mean(sc), np.std(sc),  # Spectral centroid
            np.mean(sb), np.std(sb),  # Spectral bandwidth
            np.mean(rolloff), np.std(rolloff),  # Roll-off
            np.mean(mfcc[0]), np.std(mfcc[0]),  # MFCC 1-13
            np.mean(mfcc[1]), np.mean(mfcc[2]), np.mean(mfcc[3]),
            np.mean(mfcc[4]), np.mean(mfcc[5]), np.mean(mfcc[6]),
            np.mean(mfcc[7]), np.mean(mfcc[8]), np.mean(mfcc[9]),
            np.mean(mfcc[10]),
            np.mean(voiced_f0) if len(voiced_f0) > 0 else 0,  # F0 mean
            np.std(voiced_f0) if len(voiced_f0) > 1 else 0,  # F0 std
        ]
    except Exception as e:
        return None

# Process all audio files
audio_files = glob.glob(f'{DATA_DIR}/audio/*.m4a')
print(f'Processing {len(audio_files)} audio files...')

all_features, all_labels, all_vids = [], [], []
label_mode = 'auto'  # 'auto' = pseudo-label, 'vtt' = use VTT [laughter]

for af in tqdm(audio_files):
    vid = os.path.basename(af).replace('.m4a', '')
    feat = extract_prosody(af)
    if feat is None: continue
    
    # Try to get VTT for labeling
    vtt_files = glob.glob(f'/content/drive/MyDrive/chuckle_net/vtt/{vid}.*.vtt')
    has_laugh = False
    if vtt_files:
        with open(vtt_files[0]) as f:
            content = f.read().lower()
            has_laugh = '[laughter]' in content
    
    # For now: use pseudo-label (can be refined later)
    all_features.append(feat)
    all_vids.append(vid)
    # Pseudo-label based on energy (temporary)
    all_labels.append(1 if feat[0] > 0.05 else 0)  # Crude heuristic

X = np.array(all_features)
y = np.array(all_labels)
vids = np.array(all_vids)

print(f'\nExtracted: {len(X)} samples')
print(f'Positive: {y.sum()} ({100*y.mean():.1f}%)')

# Save features
np.savez_compressed(
    f'{DATA_DIR}/prosody_features.npz',
    features=X, labels=y, vids=vids
)
print(f'Saved: {DATA_DIR}/prosody_features.npz')

In [ ]:
# === STEP 3: Extract WavLM Embeddings (GPU - SLOW) ===
# This is the bottleneck - use Colab GPU (T4/P100)

import torch
from transformers import Wav2Vec2Model
import librosa, numpy as np
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Load WavLM
print('Loading WavLM...')
wavlm = Wav2Vec2Model.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()

def extract_wavlm(audio_path, sr=16000):
    """Extract WavLM embedding for full audio"""
    try:
        y, _ = librosa.load(audio_path, sr=sr)
        if len(y) < sr: return None
        
        with torch.no_grad():
            inputs = torch.FloatTensor(y).unsqueeze(0).to(device)
            outputs = wavlm(inputs)
            emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        return emb
    except Exception as e:
        return None

# Process in batches to avoid OOM
audio_files = glob.glob(f'{DATA_DIR}/audio/*.m4a')
print(f'Processing {len(audio_files)} files with WavLM...')

all_wavlm, valid_vids = [], []
batch_size = 10

for i in tqdm(range(0, len(audio_files), batch_size):
    batch = audio_files[i:i+batch_size]
    for af in batch:
        vid = os.path.basename(af).replace('.m4a', '')
        emb = extract_wavlm(af)
        if emb is not None:
            all_wavlm.append(emb)
            valid_vids.append(vid)

X_wavlm = np.array(all_wavlm)
print(f'\nWavLM embeddings: {X_wavlm.shape}')

# Save
np.savez_compressed(
    f'{DATA_DIR}/wavlm_embeddings.npz',
    embeddings=X_wavlm, vids=np.array(valid_vids)
)
print(f'Saved: {DATA_DIR}/wavlm_embeddings.npz')

In [ ]:
# === STEP 4: Train Fusion Model ===
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load features
prosody_data = np.load(f'{DATA_DIR}/prosody_features.npz')
wavlm_data = np.load(f'{DATA_DIR}/wavlm_embeddings.npz')

X_prosody = prosody_data['features']
y = prosody_data['labels']
prosody_vids = prosody_data['vids']

# Match WavLM to prosody by video ID
wavlm_vids = wavlm_data['vids']
wavlm_emb = wavlm_data['embeddings']

# Create mapping
wavlm_map = {str(v): wavlm_emb[i] for i, v in enumerate(wavlm_vids)}
matched_wavlm = []
matched_prosody = []
matched_y = []
matched_vids = []

for i, (pv, py) in enumerate(zip(prosody_vids, y)):
    pv_str = str(pv)
    if pv_str in wavlm_map:
        matched_wavlm.append(wavlm_map[pv_str])
        matched_prosody.append(X_prosody[i])
        matched_y.append(py)
        matched_vids.append(pv_str)

X_wavlm_matched = np.array(matched_wavlm)
X_prosody_matched = np.array(matched_prosody)
y_matched = np.array(matched_y)
vids_matched = np.array(matched_vids)

print(f'Matched: {len(X_wavlm_matched)} samples')
print(f'WavLM: {X_wavlm_matched.shape}, Prosody: {X_prosody_matched.shape}')
print(f'Positive: {y_matched.sum()} ({100*y_matched.mean():.1f}%)')

# Standardize
scaler_w = StandardScaler().fit(X_wavlm_matched)
scaler_p = StandardScaler().fit(X_prosody_matched)
X_w = scaler_w.transform(X_wavlm_matched)
X_p = scaler_p.transform(X_prosody_matched)
X_fusion = np.concatenate([X_w, X_p], axis=1)

# Fusion MLP
class FusionMLP(nn.Module):
    def __init__(self, dim=791, hidden=[512, 256, 64]):
        super().__init__()
        self.bn0 = nn.BatchNorm1d(dim)
        layers = []
        prev = dim
        for h in hidden:
            layers.extend([
                nn.Linear(prev, h), nn.BatchNorm1d(h),
                nn.ReLU(), nn.Dropout(0.3)
            ])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(self.bn0(x)).squeeze(-1)

# 5-fold video-level CV
gkf = GroupKFold(n_splits=5)
f1s = []

print('\nTraining...')
for fold, (tr, te) in enumerate(gkf.split(X_fusion, y_matched, vids_matched)):
    X_tr = torch.FloatTensor(X_fusion[tr])
    y_tr = torch.FloatTensor(y_matched[tr])
    X_te = torch.FloatTensor(X_fusion[te])
    
    model = FusionMLP(dim=X_fusion.shape[1])
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)
    pos_w = torch.tensor((1-y_tr.mean())/max(0.01, y_tr.mean()))
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    
    for epoch in range(100):
        model.train()
        opt.zero_grad()
        out = model(X_tr)
        loss = loss_fn(out, y_tr)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
    
    model.eval()
    with torch.no_grad():
        pred = (torch.sigmoid(model(X_te)) > 0.5).numpy().astype(int)
    f1 = f1_score(y_matched[te], pred, zero_division=0)
    f1s.append(f1)
    print(f'  Fold {fold+1}: F1={f1:.4f} (pos={y_matched[te].mean():.1%})')

print(f'\nMean F1: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')

# Save model
torch.save(model.state_dict(), f'{DATA_DIR}/fusion_model.pt')
print(f'Model saved: {DATA_DIR}/fusion_model.pt')